# H20a: Sampled Cosine-Compounding Check

**Counterpart script:** `experiments/Tier_1_Core_Mechanism_Tests/H20a_COSINE_COMPOUNDING/run_experiment.py`

This notebook is a presentation and analysis wrapper around the script's shared
`run_experiment()` function. It does **not** reimplement the core experiment.

## Question

In this toy deep-linear setting, does **cumulative sampled Newton-alignment advantage**
track Muon's loss advantage over NormSGD better than sampled update count alone?

## Claim discipline

This notebook reports an **association test** using approximate Newton directions
(CG + finite-difference Hessian-vector products). It should not be read as a proof
of causality unless both the implemented proxy metric and the observed fitted
relationship support that interpretation.

In [ ]:
import importlib.util
import platform
import sys
import time
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np

plt.rcParams.update({
    'figure.figsize': (8, 4.5),
    'axes.grid': True,
    'grid.alpha': 0.25,
})


def locate_experiment_dir(context_path: Path):
    context_path = context_path.resolve()
    relative_script = Path('experiments/Tier_1_Core_Mechanism_Tests/H20a_COSINE_COMPOUNDING/run_experiment.py')

    direct_candidate = context_path / 'run_experiment.py'
    if direct_candidate.exists():
        return direct_candidate.parent, direct_candidate.parent.parents[3], direct_candidate

    for base in [context_path, *context_path.parents]:
        candidate = base / relative_script
        if candidate.exists():
            return candidate.parent, base, candidate

    raise FileNotFoundError('Could not locate H20a_COSINE_COMPOUNDING/run_experiment.py from the current notebook context.')


try:
    NOTEBOOK_CONTEXT = Path(__file__).resolve().parent
except NameError:
    NOTEBOOK_CONTEXT = Path.cwd()

EXPERIMENT_DIR, REPO_ROOT, SCRIPT_PATH = locate_experiment_dir(NOTEBOOK_CONTEXT)

spec = importlib.util.spec_from_file_location('h20a_cosine_compounding', SCRIPT_PATH)
h20a = importlib.util.module_from_spec(spec)
spec.loader.exec_module(h20a)

print('Notebook context :', NOTEBOOK_CONTEXT)
print('Repository root  :', REPO_ROOT)
print('Experiment dir   :', EXPERIMENT_DIR)
print('Script path      :', SCRIPT_PATH)

## Reproducibility, runtime expectations, and implemented schedule

The paired script defines the authoritative default configuration. The sampled
measurement schedule is explicit: every 50 updates, the code measures the cosine
between each optimizer's **applied momentum update** and an approximate Newton
direction at that pre-update state, then records the resulting losses **after**
the sampled update is applied.

In [ ]:
print('Python version :', platform.python_version())
print('NumPy version  :', np.__version__)
print('Matplotlib     :', plt.matplotlib.__version__)
print()

config_preview = {
    'dim': h20a.DIM,
    'n_layers': h20a.N_LAYERS,
    'num_steps': h20a.NUM_STEPS,
    'num_seeds': h20a.NUM_SEEDS,
    'batch_size': h20a.BATCH_SIZE,
    'momentum': h20a.MOMENTUM,
    'ns_iters': h20a.NS_ITERS,
    'measure_every': h20a.MEASURE_EVERY,
    'sample_steps': h20a.SAMPLE_STEPS,
    'lr_muon_candidates': h20a.LR_MUON_CANDIDATES,
    'lr_normsgd_candidates': h20a.LR_NORM_CANDIDATES,
}

for key, value in config_preview.items():
    print(f'{key:>20}: {value}')

print()
print('Notebook note: the full default run is a small toy experiment, but it still')
print('performs repeated CG/HVP computations at each sampled update.')
print('No result files are written; results remain in memory as a Python dict.')

## Execute the shared experiment

The notebook uses the script's `run_experiment()` function directly so that the
script and notebook stay aligned. The returned object contains configuration,
learning-rate sweeps, per-seed trajectories, aggregated sample summaries, fit
coefficients, and the final sign-aware verdict.

In [ ]:
start_time = time.time()
results = h20a.run_experiment(verbose=True)
elapsed_seconds = time.time() - start_time

print()
print(f'Wall-clock runtime: {elapsed_seconds:.2f} seconds')

## Structured results and basic alignment checks

These checks are intentionally lightweight. They confirm that the returned
sample schedule is internally consistent and that the notebook is consuming the
script's structured outputs rather than a duplicated local implementation.

In [ ]:
config = results['config']
sample_steps = np.array(config['sample_steps'], dtype=int)
aggregate_rows = results['aggregate_by_sample']
sample_rows = results['sample_rows']
fit_cos = results['fits']['log_loss_ratio_vs_cumulative_cosine']
fit_step = results['fits']['log_loss_ratio_vs_step']
verdict = results['verdict']

assert len(sample_steps) == config['sample_count']
assert sample_steps[0] == config['measure_every']
assert sample_steps[-1] == config['num_steps']
assert all(row['sample_step'] in set(sample_steps.tolist()) for row in sample_rows)
assert all(len(seed_result['sample_steps']) == len(sample_steps) for seed_result in results['per_seed'])

print('Basic structural checks passed.')
print('Sample schedule note:', config['sample_schedule_note'])
print('Chosen LRs:', results['lr_sweeps']['best_lr_muon'], '(Muon),', results['lr_sweeps']['best_lr_normsgd'], '(NormSGD)')
print('Returned sample rows:', len(sample_rows))
print('Seeds represented   :', results['seeds'])

## Learning-rate selection on the first seed

This is a convenience sweep rather than a full hyperparameter study. Each
optimizer gets its own candidate grid, and the chosen learning rate is the one
with the lowest final loss on the first seed.

In [ ]:
muon_lr_records = results['lr_sweeps']['muon']
norm_lr_records = results['lr_sweeps']['normsgd']

muon_lrs = np.array([row['lr'] for row in muon_lr_records], dtype=float)
muon_final_losses = np.array([row['final_loss'] for row in muon_lr_records], dtype=float)
norm_lrs = np.array([row['lr'] for row in norm_lr_records], dtype=float)
norm_final_losses = np.array([row['final_loss'] for row in norm_lr_records], dtype=float)

plt.figure(figsize=(8, 4.5))
plt.plot(muon_lrs, muon_final_losses, marker='o', label='Muon')
plt.plot(norm_lrs, norm_final_losses, marker='s', label='NormSGD')
plt.xscale('log')
plt.yscale('log')
plt.xlabel('Learning rate candidate')
plt.ylabel('Final training loss on sweep seed')
plt.title('Learning-rate sweep used by the paired script')
plt.legend()
plt.show()

print('Best Muon LR    :', results['lr_sweeps']['best_lr_muon'])
print('Best NormSGD LR :', results['lr_sweeps']['best_lr_normsgd'])

**Interpretation.** The learning-rate sweep is part of the experimental setup,
not part of the final cosine-compounding claim. It reduces a trivial fairness
issue (using obviously bad LRs) but does not eliminate broader tuning concerns.

## Sampled cosine measurements over sampled update count

The left panel shows the **sampled cosine advantage** itself at each labeled
sample step. The right panel shows the **cumulative sampled cosine advantage**,
which is the explanatory variable used in the main association fit.

In [ ]:
agg_steps = np.array([row['sample_step'] for row in aggregate_rows], dtype=float)
mean_cos_adv = np.array([row['mean_sampled_cosine_advantage'] for row in aggregate_rows], dtype=float)
std_cos_adv = np.array([row['std_sampled_cosine_advantage'] for row in aggregate_rows], dtype=float)
mean_cumul_cos = np.array([row['mean_cumulative_sampled_cosine_advantage'] for row in aggregate_rows], dtype=float)
std_cumul_cos = np.array([row['std_cumulative_sampled_cosine_advantage'] for row in aggregate_rows], dtype=float)

fig, axes = plt.subplots(1, 2, figsize=(12, 4.5))

for seed_result in results['per_seed']:
    axes[0].plot(seed_result['sample_steps'], seed_result['sampled_cosine_advantage'], color='tab:blue', alpha=0.15)
    axes[1].plot(seed_result['sample_steps'], seed_result['sampled_cumulative_cosine_advantage'], color='tab:green', alpha=0.15)

axes[0].errorbar(agg_steps, mean_cos_adv, yerr=std_cos_adv, marker='o', color='tab:blue', capsize=3)
axes[0].axhline(0.0, color='black', linewidth=1, alpha=0.5)
axes[0].set_title('Sampled cosine advantage')
axes[0].set_xlabel('Completed update count (sample step)')
axes[0].set_ylabel('cos(Muon, Newton) - cos(NormSGD, Newton)')

axes[1].errorbar(agg_steps, mean_cumul_cos, yerr=std_cumul_cos, marker='o', color='tab:green', capsize=3)
axes[1].axhline(0.0, color='black', linewidth=1, alpha=0.5)
axes[1].set_title('Cumulative sampled cosine advantage')
axes[1].set_xlabel('Completed update count (sample step)')
axes[1].set_ylabel('Cumulative sampled cosine advantage')

plt.tight_layout()
plt.show()

print('First sampled mean cosine advantage :', mean_cos_adv[0])
print('Last sampled mean cosine advantage  :', mean_cos_adv[-1])
print('Final mean cumulative cosine        :', mean_cumul_cos[-1])

**Interpretation.** This figure answers a narrow descriptive question: whether
the sampled Newton-alignment advantage is persistently positive, noisy, or mixed
across the trajectory. A cumulative curve can drift upward even if individual
sampled advantages are small, but a later negative drift or high seed variance
weakens any strong compounding story.

## Loss-ratio trajectory over sampled update count

The outcome variable is the sampled post-update loss ratio
$\mathrm{loss}_{\mathrm{NormSGD}} / \mathrm{loss}_{\mathrm{Muon}}$. Values above 1
favor Muon. A logarithmic y-axis is useful because the ratio can vary over more
than an order of magnitude.

In [ ]:
mean_loss_ratio = np.array([row['mean_loss_ratio'] for row in aggregate_rows], dtype=float)
std_loss_ratio = np.array([row['std_loss_ratio'] for row in aggregate_rows], dtype=float)

plt.figure(figsize=(8, 4.5))
for seed_result in results['per_seed']:
    plt.plot(seed_result['sample_steps'], seed_result['sampled_loss_ratio'], color='tab:orange', alpha=0.18)
plt.errorbar(agg_steps, mean_loss_ratio, yerr=std_loss_ratio, marker='o', color='tab:red', capsize=3, label='mean ± std across seeds')
plt.axhline(1.0, color='black', linewidth=1, alpha=0.5)
plt.yscale('log')
plt.xlabel('Completed update count (sample step)')
plt.ylabel('loss_NormSGD / loss_Muon')
plt.title('Sampled loss-ratio trajectory')
plt.legend()
plt.show()

print('Final sampled mean loss ratio:', mean_loss_ratio[-1])
print('Final sampled std loss ratio :', std_loss_ratio[-1])

**Interpretation.** The ratio trajectory shows whether Muon's advantage grows,
shrinks, or changes sign across the sampled updates. The causal interpretation,
however, still depends on whether the cosine-based explanatory variable behaves
in the expected direction and improves on a simple time-based comparator.

## Main association view: `log(loss ratio)` vs cumulative sampled cosine advantage

This is the notebook's primary fit. The regression is done on the returned
**per-seed sampled rows**, not just on checkpoint means, so each seed contributes
one data point per sampled update.

In [ ]:
x_cos = np.array(fit_cos['x'], dtype=float)
y_cos = np.array(fit_cos['y'], dtype=float)
y_cos_pred = np.array(fit_cos['predicted_y'], dtype=float)

agg_x_cos = np.array([row['mean_cumulative_sampled_cosine_advantage'] for row in aggregate_rows], dtype=float)
agg_y_log_ratio = np.array([row['mean_log_loss_ratio'] for row in aggregate_rows], dtype=float)

order = np.argsort(x_cos)

plt.figure(figsize=(7, 5))
plt.scatter(x_cos, y_cos, alpha=0.45, label='Per-seed sampled rows')
plt.scatter(agg_x_cos, agg_y_log_ratio, color='black', s=60, label='Sample means by sample step')
if fit_cos['valid']:
    plt.plot(x_cos[order], y_cos_pred[order], color='tab:red', linewidth=2, label='Linear fit')
plt.xlabel('Cumulative sampled cosine advantage')
plt.ylabel('log(loss_NormSGD / loss_Muon)')
plt.title('Association between cumulative sampled cosine and loss advantage')
plt.legend()
plt.show()

print('Cosine-fit slope     :', fit_cos['slope'])
print('Cosine-fit intercept :', fit_cos['intercept'])
print('Cosine-fit R^2       :', fit_cos['r2'])
print('Cosine-fit n_points  :', fit_cos['n_points'])

**Interpretation.** For the compounding story to be supported even in this
limited proxy sense, the fitted slope should be **positive**: more cumulative
sampled cosine advantage should correspond to a larger Muon loss advantage.
A higher $R^2$ alone is not enough if the fitted slope points in the wrong
direction.

## Comparator: `log(loss ratio)` vs sampled update count

This comparator asks whether the observed divergence is already explained about
as well by time alone. If the step-only fit performs similarly, then the cosine
proxy is not adding much explanatory power in this first-pass toy analysis.

In [ ]:
x_step = np.array(fit_step['x'], dtype=float)
y_step = np.array(fit_step['y'], dtype=float)
y_step_pred = np.array(fit_step['predicted_y'], dtype=float)

order = np.argsort(x_step)

plt.figure(figsize=(7, 5))
plt.scatter(x_step, y_step, alpha=0.45, label='Per-seed sampled rows')
plt.scatter(agg_steps, agg_y_log_ratio, color='black', s=60, label='Sample means by sample step')
if fit_step['valid']:
    plt.plot(x_step[order], y_step_pred[order], color='tab:purple', linewidth=2, label='Linear fit')
plt.xlabel('Completed update count (sample step)')
plt.ylabel('log(loss_NormSGD / loss_Muon)')
plt.title('Temporal comparator fit')
plt.legend()
plt.show()

print('Step-fit slope     :', fit_step['slope'])
print('Step-fit intercept :', fit_step['intercept'])
print('Step-fit R^2       :', fit_step['r2'])
print('Step-fit n_points  :', fit_step['n_points'])
print('Delta R^2 (cos - step):', verdict['delta_r2'])

## Summary tables

The following cell prints compact textual summaries for reproducibility: chosen
learning rates, fit coefficients, and the final sampled values for each seed.

In [ ]:
print('=== Learning-rate choices ===')
print('Sweep seed        :', results['lr_sweeps']['seed'])
print('Best Muon LR      :', results['lr_sweeps']['best_lr_muon'])
print('Best NormSGD LR   :', results['lr_sweeps']['best_lr_normsgd'])
print()

print('=== Fit summary ===')
print(f"{'Model':<48} {'Slope':>12} {'Intercept':>12} {'R^2':>10} {'n':>6}")
print('-' * 92)
print(f"{'log(loss ratio) ~ cumulative sampled cosine':<48} {fit_cos['slope']:>12.6f} {fit_cos['intercept']:>12.6f} {fit_cos['r2']:>10.4f} {fit_cos['n_points']:>6}")
print(f"{'log(loss ratio) ~ sampled step count':<48} {fit_step['slope']:>12.6f} {fit_step['intercept']:>12.6f} {fit_step['r2']:>10.4f} {fit_step['n_points']:>6}")
print()

print('=== Per-seed final sampled values ===')
print(f"{'Seed':>8} {'Final ratio':>14} {'Final cumul cos':>18} {'Valid samples':>14}")
print('-' * 62)
for seed_result in results['per_seed']:
    print(
        f"{seed_result['seed']:>8} "
        f"{seed_result['final_loss_ratio']:>14.6f} "
        f"{seed_result['final_cumulative_cosine_advantage']:>18.6f} "
        f"{seed_result['valid_sample_count']:>14}"
    )

## Final verdict and caveats

The verdict below explicitly checks the sign of the cosine-based slope. This is
the main guardrail added in the first pair-level completion pass: a negative
cosine slope must **not** be reported as support for cosine compounding.

In [ ]:
print('=== Final verdict ===')
print('Status                    :', verdict['status'])
print('Supports compounding?     :', verdict['supports_compounding'])
print('Cosine slope positive?    :', verdict['cosine_slope_positive'])
print('Cosine fit beats step fit?:', verdict['cosine_fit_beats_step_fit'])
print('Delta R^2 (cos - step)    :', verdict['delta_r2'])
print('Message                   :', verdict['message'])
print()
print('Implemented metric reminder:')
print('-', config['sample_schedule_note'])
print()
print('Important caveats:')
print('- approximate Newton directions come from CG with finite-difference HVPs;')
print('- this is a toy deep-linear system, not a realistic large-model training run;')
print('- sampled rows within a seed are temporally correlated;')
print('- learning rates are chosen from a small sweep on the first seed only;')
print('- the current notebook tests association, not causal mechanism identification.')